In [25]:
import numpy as np
import os
import pandas as pd
import pickle
from tqdm import tqdm
project_dir = os.path.expanduser('../')
load_dir_vf = project_dir + "/data/value_functions"
load_dir_π = project_dir + "/data/policies"
from sklearn.utils import resample


In [26]:
SA_mask = pd.read_pickle(project_dir + "data/behavior_policy/SA_mask_shifted.pkl")
if isinstance(SA_mask, pd.DataFrame):
    SA_mask = SA_mask.values
    
for fname in sorted(os.listdir(load_dir_vf)):
    if fname.endswith(".npz"):
        var_name = fname.replace(".npz", "")
        globals()[var_name] = np.load(os.path.join(load_dir_vf, fname))["data"]

for fname in sorted(os.listdir(load_dir_π)):
    if fname.endswith(".pkl"):
        var_name = fname.replace(".pkl", "")
        with open(os.path.join(load_dir_π, fname), "rb") as f:
            globals()[var_name] = pickle.load(f)

with open(project_dir + 'data/env_model/gymP_shifted_plus.pkl', 'rb') as f:
    gymP_plus = pickle.load(f)
with open(project_dir + 'data/env_model/gymP_shifted_minus.pkl', 'rb') as f:
    gymP_minus = pickle.load(f)
with open(project_dir + 'data/env_model/gymP_shifted_mixed.pkl', 'rb') as f:
    gymP_mixed = pickle.load(f)
    
traj_shifted_test = pd.read_csv(project_dir + "data/traj_shifted_test.csv")    
    
γ = 1

nS = 750
nA = 25

nS_term = 2
S_survival = 750
S_death = 751

nS_total = nS + nS_term

### Checking Propositions
#### Checking Proposition 2
Proposition 2 holds, which is expected (proposition 2 doesn't require guaranteed absorption)

V_plus + V_minus = V_mixed


In [27]:
np.allclose(V_plus + V_minus, V_mixed, atol=1e-6, rtol=0)

True

Q_plus + Q_minus = Q_mixed

In [28]:
valid_mask = ~np.isnan(Q_plus) & ~np.isnan(Q_minus) & ~np.isnan(Q_mixed)
diff = np.abs((Q_plus + Q_minus) - Q_mixed)
print(np.all(diff[valid_mask] <= 1e-6))

True


#### Checking Proposition 4

V_plus - V_minus = 1

In [29]:
np.allclose(V_plus - V_minus, 1.0, atol=1e-7, rtol=0)
np.where(~np.isclose(V_plus - V_minus, 1.0, atol=1e-7, rtol=0))

(array([271, 330, 482]),)

In [30]:
for target_states in [271, 330, 482]:
    print(V_plus[target_states] - V_minus[target_states])

0.9999998590580437
0.0
0.0


In [31]:
gymP_minus[330]

defaultdict(list,
            {0: [(np.float64(1.0), 582, np.float64(0.0), 0)],
             5: [(np.float64(0.25), 82, np.float64(0.0), 0),
              (np.float64(0.5), 330, np.float64(0.0), 0),
              (np.float64(0.25), 511, np.float64(0.0), 0)],
             6: [(np.float64(1.0), 330, np.float64(0.0), 0)],
             7: [(np.float64(0.75), 330, np.float64(0.0), 0),
              (np.float64(0.25), 751, np.float64(-1.0), 1)],
             8: [(np.float64(0.75), 330, np.float64(0.0), 0),
              (np.float64(0.25), 362, np.float64(0.0), 0)],
             11: [(np.float64(1.0), 330, np.float64(0.0), 0)],
             12: [(np.float64(1.0), 330, np.float64(0.0), 0)],
             13: [(np.float64(1.0), 330, np.float64(0.0), 0)],
             16: [(np.float64(1.0), 330, np.float64(0.0), 0)],
             17: [(np.float64(1.0), 330, np.float64(0.0), 0)],
             18: [(np.float64(1.0), 330, np.float64(0.0), 0)],
             22: [(np.float64(1.0), 330, np.float64(0.0)

In [32]:
gymP_plus[330]

defaultdict(list,
            {0: [(np.float64(1.0), 582, np.float64(0.0), 0)],
             5: [(np.float64(0.25), 82, np.float64(0.0), 0),
              (np.float64(0.5), 330, np.float64(0.0), 0),
              (np.float64(0.25), 511, np.float64(0.0), 0)],
             6: [(np.float64(1.0), 330, np.float64(0.0), 0)],
             7: [(np.float64(0.75), 330, np.float64(0.0), 0),
              (np.float64(0.25), 751, np.float64(0.0), 1)],
             8: [(np.float64(0.75), 330, np.float64(0.0), 0),
              (np.float64(0.25), 362, np.float64(0.0), 0)],
             11: [(np.float64(1.0), 330, np.float64(0.0), 0)],
             12: [(np.float64(1.0), 330, np.float64(0.0), 0)],
             13: [(np.float64(1.0), 330, np.float64(0.0), 0)],
             16: [(np.float64(1.0), 330, np.float64(0.0), 0)],
             17: [(np.float64(1.0), 330, np.float64(0.0), 0)],
             18: [(np.float64(1.0), 330, np.float64(0.0), 0)],
             22: [(np.float64(1.0), 330, np.float64(0.0),

In [33]:
Q_plus[330]

array([nan, nan, nan, nan, nan, nan, nan,  0., nan, nan, nan, nan, nan,
       nan, nan, nan, nan, nan,  0., nan, nan, nan, nan, nan, nan])

In [34]:
Q_minus[330]

array([  nan,   nan,   nan,   nan,   nan,   nan,   nan, -0.25,   nan,
         nan,   nan,   nan,   nan,   nan,   nan,   nan,   nan,   nan,
        0.  ,   nan,   nan,   nan,   nan,   nan,   nan])

In [35]:
π_minus[330]

[18]

#### Checking Proposition 5

Q_plus - Q_minus = 1

In [36]:
tol = 1e-6
valid_mask = ~np.isnan(Q_plus) & ~np.isnan(Q_minus)
diff = (Q_plus - Q_minus) - 1.0
fail_mask = valid_mask & (np.abs(diff) > tol)
bad_indices = np.argwhere(fail_mask)
n_valid = np.count_nonzero(valid_mask)
n_fail  = len(bad_indices)
print(f"Checked {n_valid} valid (s,a) pairs → {n_fail} violated Proposition 4 "
      f"(Q⁺−Q⁻ ≈ 1).")

for s, a in bad_indices:
    print(f"State {s:3d}, Action {a:2d} | "
          f"Q⁺={Q_plus[s,a]:.6f}, Q⁻={Q_minus[s,a]:.6f}, "
          f"Δ={Q_plus[s,a]-Q_minus[s,a]:.6f}")


Checked 3834 valid (s,a) pairs → 5 violated Proposition 4 (Q⁺−Q⁻ ≈ 1).
State  82, Action  5 | Q⁺=0.934426, Q⁻=-0.049180, Δ=0.983607
State 116, Action  5 | Q⁺=0.962963, Q⁻=-0.000000, Δ=0.962963
State 330, Action  7 | Q⁺=0.000000, Q⁻=-0.250000, Δ=0.250000
State 330, Action 18 | Q⁺=0.000000, Q⁻=0.000000, Δ=0.000000
State 482, Action  9 | Q⁺=0.000000, Q⁻=0.000000, Δ=0.000000


In [37]:
gymP_plus[82]

defaultdict(list,
            {0: [(np.float64(0.47619047619047616), 82, np.float64(0.0), 0),
              (np.float64(0.047619047619047616), 197, np.float64(0.0), 0),
              (np.float64(0.09523809523809523), 202, np.float64(0.0), 0),
              (np.float64(0.047619047619047616), 314, np.float64(0.0), 0),
              (np.float64(0.047619047619047616), 416, np.float64(0.0), 0),
              (np.float64(0.047619047619047616), 515, np.float64(0.0), 0),
              (np.float64(0.047619047619047616), 547, np.float64(0.0), 0),
              (np.float64(0.047619047619047616), 646, np.float64(0.0), 0),
              (np.float64(0.047619047619047616), 681, np.float64(0.0), 0),
              (np.float64(0.09523809523809523), 750, np.float64(1.0), 1)],
             5: [(np.float64(0.01639344262295082), 35, np.float64(0.0), 0),
              (np.float64(0.01639344262295082), 59, np.float64(0.0), 0),
              (np.float64(0.01639344262295082), 66, np.float64(0.0), 0),
          

#### Comparing policies
In proposition 6, we expected when all three assumptions hold, all policies should be equivalent.

**gamma = 1**

π_plus vs. π_minus

In [38]:
def compare_policy(π1, π2, name_π1, name_π2):
    count = 0
    for s in range(nS):
        if not np.array_equal(π1[s], π2[s]):
            count += 1
            print(f"Different policies between {name_π1} and {name_π2} at state {s}")
            print(f"{name_π1}: ", π1[s])
            print(f"{name_π2}: ", π2[s])
    print("Number of states with different policy: ", count)

In [39]:
π_plus[330]

[7, 18]

In [40]:
π_plus_gamma_99[330]

[7, 18]

In [41]:
# Comparing π_plus and π_minus, print different states;
compare_policy(π_plus, π_minus, "π_plus", "π_minus")

Different policies between π_plus and π_minus at state 116
π_plus:  [0]
π_minus:  [0, 5]
Different policies between π_plus and π_minus at state 330
π_plus:  [7, 18]
π_minus:  [18]
Number of states with different policy:  2


π_plus vs. π_mixed

In [42]:
# Comparing π_plus and π_minus, print different states;
compare_policy(π_plus, π_mixed, "π_plus", "π_mixed")

Different policies between π_plus and π_mixed at state 51
π_plus:  [5, 8, 10]
π_mixed:  [5, 8]
Different policies between π_plus and π_mixed at state 294
π_plus:  [10, 15]
π_mixed:  [15]
Different policies between π_plus and π_mixed at state 330
π_plus:  [7, 18]
π_mixed:  [18]
Different policies between π_plus and π_mixed at state 478
π_plus:  [5, 10, 14, 15]
π_mixed:  [10, 14, 15]
Different policies between π_plus and π_mixed at state 569
π_plus:  [0, 1]
π_mixed:  [1]
Number of states with different policy:  5


π_minus vs. π_mixed

In [43]:
# Comparing π_minus and π_mixed, print different states;
compare_policy(π_minus, π_mixed, "π_minus", "π_mixed")

Different policies between π_minus and π_mixed at state 51
π_minus:  [5, 8, 10]
π_mixed:  [5, 8]
Different policies between π_minus and π_mixed at state 116
π_minus:  [0, 5]
π_mixed:  [0]
Different policies between π_minus and π_mixed at state 294
π_minus:  [10, 15]
π_mixed:  [15]
Different policies between π_minus and π_mixed at state 478
π_minus:  [5, 10, 14, 15]
π_mixed:  [10, 14, 15]
Different policies between π_minus and π_mixed at state 569
π_minus:  [0, 1]
π_mixed:  [1]
Number of states with different policy:  5


**gamma = 0.99**

In [44]:
# Comparing π_plus and π_minus, print different states;
compare_policy(π_plus_gamma_99, π_minus_gamma_99, "π_plus_gamma_99", "π_minus_gamma_99")

Different policies between π_plus_gamma_99 and π_minus_gamma_99 at state 0
π_plus_gamma_99:  [0]
π_minus_gamma_99:  [0, 6, 7, 8, 10, 15]
Different policies between π_plus_gamma_99 and π_minus_gamma_99 at state 1
π_plus_gamma_99:  [0]
π_minus_gamma_99:  [0, 5, 10, 15]
Different policies between π_plus_gamma_99 and π_minus_gamma_99 at state 2
π_plus_gamma_99:  [0]
π_minus_gamma_99:  [0, 6, 7, 8, 10]
Different policies between π_plus_gamma_99 and π_minus_gamma_99 at state 3
π_plus_gamma_99:  [8]
π_minus_gamma_99:  [6, 8, 9, 14, 15]
Different policies between π_plus_gamma_99 and π_minus_gamma_99 at state 4
π_plus_gamma_99:  [8]
π_minus_gamma_99:  [0, 7, 8, 9, 10, 15]
Different policies between π_plus_gamma_99 and π_minus_gamma_99 at state 5
π_plus_gamma_99:  [0]
π_minus_gamma_99:  [6, 8, 9, 15]
Different policies between π_plus_gamma_99 and π_minus_gamma_99 at state 6
π_plus_gamma_99:  [0]
π_minus_gamma_99:  [0, 5, 7, 10, 15]
Different policies between π_plus_gamma_99 and π_minus_gamma_99 

In [45]:
# Comparing π_plus and π_minus, print different states;
compare_policy(π_plus_gamma_99, π_mixed_gamma_99, "π_plus_gamma_99", "π_mixed__gamma_99")

Different policies between π_plus_gamma_99 and π_mixed__gamma_99 at state 41
π_plus_gamma_99:  [5]
π_mixed__gamma_99:  [0]
Different policies between π_plus_gamma_99 and π_mixed__gamma_99 at state 65
π_plus_gamma_99:  [0]
π_mixed__gamma_99:  [15]
Different policies between π_plus_gamma_99 and π_mixed__gamma_99 at state 106
π_plus_gamma_99:  [0]
π_mixed__gamma_99:  [6]
Different policies between π_plus_gamma_99 and π_mixed__gamma_99 at state 167
π_plus_gamma_99:  [10]
π_mixed__gamma_99:  [5]
Different policies between π_plus_gamma_99 and π_mixed__gamma_99 at state 179
π_plus_gamma_99:  [0]
π_mixed__gamma_99:  [6]
Different policies between π_plus_gamma_99 and π_mixed__gamma_99 at state 246
π_plus_gamma_99:  [5]
π_mixed__gamma_99:  [15]
Different policies between π_plus_gamma_99 and π_mixed__gamma_99 at state 308
π_plus_gamma_99:  [0]
π_mixed__gamma_99:  [7]
Different policies between π_plus_gamma_99 and π_mixed__gamma_99 at state 320
π_plus_gamma_99:  [0]
π_mixed__gamma_99:  [9]
Differe

In [46]:
# Comparing π_minus and π_mixed, print different states;
compare_policy(π_minus_gamma_99, π_mixed_gamma_99, "π_minus_gamma_99", "π_mixed__gamma_99")

Different policies between π_minus_gamma_99 and π_mixed__gamma_99 at state 0
π_minus_gamma_99:  [0, 6, 7, 8, 10, 15]
π_mixed__gamma_99:  [0]
Different policies between π_minus_gamma_99 and π_mixed__gamma_99 at state 1
π_minus_gamma_99:  [0, 5, 10, 15]
π_mixed__gamma_99:  [0]
Different policies between π_minus_gamma_99 and π_mixed__gamma_99 at state 2
π_minus_gamma_99:  [0, 6, 7, 8, 10]
π_mixed__gamma_99:  [0]
Different policies between π_minus_gamma_99 and π_mixed__gamma_99 at state 3
π_minus_gamma_99:  [6, 8, 9, 14, 15]
π_mixed__gamma_99:  [8]
Different policies between π_minus_gamma_99 and π_mixed__gamma_99 at state 4
π_minus_gamma_99:  [0, 7, 8, 9, 10, 15]
π_mixed__gamma_99:  [8]
Different policies between π_minus_gamma_99 and π_mixed__gamma_99 at state 5
π_minus_gamma_99:  [6, 8, 9, 15]
π_mixed__gamma_99:  [0]
Different policies between π_minus_gamma_99 and π_mixed__gamma_99 at state 6
π_minus_gamma_99:  [0, 5, 7, 10, 15]
π_mixed__gamma_99:  [0]
Different policies between π_minus_g

#### Comparing policies between gamma = 1 and gamma = 0.99

In [47]:
compare_policy(π_plus, π_plus_gamma_99, "π_plus", "π_plus_gamma_99")

Different policies between π_plus and π_plus_gamma_99 at state 0
π_plus:  [0, 6, 7, 8, 10, 15]
π_plus_gamma_99:  [0]
Different policies between π_plus and π_plus_gamma_99 at state 1
π_plus:  [0, 5, 10, 15]
π_plus_gamma_99:  [0]
Different policies between π_plus and π_plus_gamma_99 at state 2
π_plus:  [0, 6, 7, 8, 10]
π_plus_gamma_99:  [0]
Different policies between π_plus and π_plus_gamma_99 at state 3
π_plus:  [6, 8, 9, 14, 15]
π_plus_gamma_99:  [8]
Different policies between π_plus and π_plus_gamma_99 at state 4
π_plus:  [0, 7, 8, 9, 10, 15]
π_plus_gamma_99:  [8]
Different policies between π_plus and π_plus_gamma_99 at state 5
π_plus:  [6, 8, 9, 15]
π_plus_gamma_99:  [0]
Different policies between π_plus and π_plus_gamma_99 at state 6
π_plus:  [0, 5, 7, 10, 15]
π_plus_gamma_99:  [0]
Different policies between π_plus and π_plus_gamma_99 at state 7
π_plus:  [0, 6, 7, 9, 10, 15]
π_plus_gamma_99:  [0]
Different policies between π_plus and π_plus_gamma_99 at state 8
π_plus:  [0, 5, 8, 10,

In [48]:
compare_policy(π_minus, π_minus_gamma_99, "π_minus", "π_minus_gamma_99")

Different policies between π_minus and π_minus_gamma_99 at state 635
π_minus:  [0, 10, 15]
π_minus_gamma_99:  [0, 7, 10, 15]
Number of states with different policy:  1


In [49]:
compare_policy(π_mixed, π_mixed_gamma_99, "π_mixed", "π_mixed_gamma_99")

Different policies between π_mixed and π_mixed_gamma_99 at state 0
π_mixed:  [0, 6, 7, 8, 10, 15]
π_mixed_gamma_99:  [0]
Different policies between π_mixed and π_mixed_gamma_99 at state 1
π_mixed:  [0, 5, 10, 15]
π_mixed_gamma_99:  [0]
Different policies between π_mixed and π_mixed_gamma_99 at state 2
π_mixed:  [0, 6, 7, 8, 10]
π_mixed_gamma_99:  [0]
Different policies between π_mixed and π_mixed_gamma_99 at state 3
π_mixed:  [6, 8, 9, 14, 15]
π_mixed_gamma_99:  [8]
Different policies between π_mixed and π_mixed_gamma_99 at state 4
π_mixed:  [0, 7, 8, 9, 10, 15]
π_mixed_gamma_99:  [8]
Different policies between π_mixed and π_mixed_gamma_99 at state 5
π_mixed:  [6, 8, 9, 15]
π_mixed_gamma_99:  [0]
Different policies between π_mixed and π_mixed_gamma_99 at state 6
π_mixed:  [0, 5, 7, 10, 15]
π_mixed_gamma_99:  [0]
Different policies between π_mixed and π_mixed_gamma_99 at state 7
π_mixed:  [0, 6, 7, 9, 10, 15]
π_mixed_gamma_99:  [0]
Different policies between π_mixed and π_mixed_gamma_99

#### Quick Summaries on Differences

**γ = 1**

| Comparison | # of Different States | 
|-------------|----------------------|
| π⁺ vs. π⁻ | **2** |  
| π⁺ vs. π+- | **5** |  
| π⁻ vs. π+- | **5** |  

**γ = 0.99**

| Comparison | # of Different States | 
|-------------|----------------------|
| π⁺ vs. π⁻ | **700** |  
| π⁺ vs. π+- | **20** |  
| π⁻ vs. π+- | **695** |  


**γ = 1 → γ = 0.99**

| Policy | # of Changed States | 
|---------|--------------------|
| π⁺ | **698** |  
| π⁻ | **1** |  
| π+- | **694** |  |


3 interesting findings:
- for gamma = 1, intra policy difference is small. for gamma = 0.99, intra policy difference is big (π+ vs. π+- is relatively small). Expected; temporality.
- for gamma = 0.99, intra policy difference is relatively big but not for π+ vs. π+-. Expected; π- adds only death avoidance which is already encoded in π+.
- when gamma changes from 1 -> 0.99, π+ and π+- changed dramatically (nS almost 700), but π- almost did not change. Expected; gamma = 0.99 introduces temporality, which break former ties by prioritizing faster recovery.

To conclude, we can't confidently conclude that π+ is the "best". But it does optimizes discounted time to survival, which is survival oriented + temporality discounted, which might be useful in some cases. Same thing applies to gamma, use 0.99 if we want to include faster recovery, gamma = 1 if we only care about eventual outcome, regardless of when it happens.<br>
e.g. In chronic disease treatment, the trajectory matter as much as the endpoint, and faster recovery is not "always preferred".

##### Other Metrics (TBD)

**policy density..?** (density of optimal actions per state). Higher percentage means the policy is "less decisive" and gives more "general policy" e.g. in gridworld, one policy goes in all four directions

In [50]:
# total (s, a) in π / total unmasked (s, a)
def policy_density(π):
    count = 0
    for policies in π.values():
        count += len(policies)
    return count * 100 / np.sum(np.array(SA_mask)) 

print(policy_density(π_plus))
print(policy_density(π_minus))
print(policy_density(π_mixed))
print(policy_density(π_plus_gamma_99))
print(policy_density(π_minus_gamma_99))
print(policy_density(π_mixed_gamma_99))

73.73500260824204
73.73500260824204
73.60459050599896
19.58789775691184
73.76108502869066
19.561815336463223


$\gamma = 1$ gives more loose policy (~73%) while $\gamma = 0.99$ gives more decisive policy excluding π_minus_gamma_99, which is similar to gamma = 1.

**survival rate per policy**

In [51]:
# converting dict policies to sparse matrices
# @TODO: fix prior steps, saving all π as sparse matrices

policies = {
    "π_plus": π_plus,
    "π_minus": π_minus,
    "π_mixed": π_mixed,
    "π_plus_gamma_99": π_plus_gamma_99,
    "π_minus_gamma_99": π_minus_gamma_99,
    "π_mixed_gamma_99": π_mixed_gamma_99
}

for name, π in policies.items():
    π_matrix = np.zeros((750, 25))
    for s, a in π.items():
        if not (0 <= s < 750):
            continue
        if isinstance(a, (int, np.integer)):
            if 0 <= a < 25:
                π_matrix[s, a] = 1.0
        elif isinstance(a, (list, np.ndarray, set)):
            valid_actions = [int(x) for x in a if 0 <= x < 25]
            if valid_actions:
                prob = 1.0 / len(valid_actions)
                for act in valid_actions:
                    π_matrix[s, act] = prob
    policies[name] = π_matrix  # replace dict with np.ndarray

# unpack back to named variables
π_plus            = policies["π_plus"]
π_minus           = policies["π_minus"]
π_mixed           = policies["π_mixed"]
π_plus_gamma_99   = policies["π_plus_gamma_99"]
π_minus_gamma_99  = policies["π_minus_gamma_99"]
π_mixed_gamma_99  = policies["π_mixed_gamma_99"]

OPE - direct (value iteration in MDP+, then get average V score)

In [52]:
import numpy as np
from tqdm import trange

def iter_policy_eval(gymP, policy, mask_valid=None, tol=1e-8, max_iter=5000):
    # puts π in MDP+ with gamma = 1, and average out the V+
    nS_transient = len(gymP)
    nS = nS_transient + 2
    RECOVERY, DEATH = nS_transient, nS_transient + 1

    x = np.zeros(nS)
    x[RECOVERY] = 1.0
    if mask_valid is None:
        mask_valid = np.ones(nS_transient, dtype=bool)

    for _ in trange(max_iter, desc="Iterating recovery probability", leave=False):
        delta = 0.0
        x_new = x.copy()

        for s in range(nS_transient):
            if not mask_valid[s]:
                continue

            elif isinstance(policy, np.ndarray):
                state_prob = 0.0
                for a, a_prob in enumerate(policy[s]):
                    if a_prob <= 0 or a not in gymP.get(s, {}):
                        continue
                    state_prob += a_prob * sum(float(p) * x[int(s_next)] for (p, s_next, _, _) in gymP[s][a])
                x_new[s] = state_prob

            delta = max(delta, abs(x_new[s] - x[s]))

        x = x_new
        if delta < tol:
            # print(f"Converged after {it+1} iterations (Δ={delta:.2e})")
            break
    else:
        print(f"Reached max_iter ({max_iter}) without full convergence (Δ={delta:.2e})")

    return x

π_b_shifted = pd.read_pickle(project_dir + 'data/behavior_policy/π_b_shifted.pkl').values
policies.update({"π_b_shifted": π_b_shifted})

variants = {
    "MDP_plus": gymP_plus,  
    "MDP_minus": gymP_minus,
    "MDP_mixed": gymP_mixed 
}

results = {}
mask_valid = SA_mask.any(axis=1)
for variant_name, gymP in variants.items():
    print(f"\nEvaluating under {variant_name}...")
    for π_name, π in policies.items():
        survival_prob = iter_policy_eval(gymP, π, mask_valid)
        expected_value = np.mean(survival_prob[:750][mask_valid[:750]])
        results[(variant_name, π_name)] = expected_value
        print(f"  Expected value ({π_name}): {expected_value:.4f}")

results_df = (
    pd.DataFrame(results.values(), 
                 index=pd.MultiIndex.from_tuples(results.keys(), names=["MDP", "Policy"]), 
                 columns=["Expected_Value"])
    .reset_index()
    .pivot(index="Policy", columns="MDP", values="Expected_Value")
)
print("\nSummary Table:\n", results_df.round(4))



Evaluating under MDP_plus...


  Expected value (π_plus): 0.9950


  Expected value (π_minus): 0.9948


  Expected value (π_mixed): 0.9950


  Expected value (π_plus_gamma_99): 0.9939


...

  Expected value (π_mixed): 0.9950


  Expected value (π_plus_gamma_99): 0.9939


  Expected value (π_minus_gamma_99): 0.9948


  Expected value (π_mixed_gamma_99): 0.9946


  Expected value (π_b_shifted): 0.8862

Summary Table:
 MDP               MDP_minus  MDP_mixed  MDP_plus
Policy                                          
π_b_shifted          0.8862     0.8862    0.8862
π_minus              0.9948     0.9948    0.9948
π_minus_gamma_99     0.9948     0.9948    0.9948
π_mixed              0.9950     0.9950    0.9950
π_mixed_gamma_99     0.9946     0.9946    0.9946
π_plus               0.9950     0.9950    0.9950
π_plus_gamma_99      0.9939     0.9939    0.9939


OPE - IS, WIS, DR, and WDR

In [53]:
def IS(trajectories, π_e, π_b, clip_rho=None, clip_value=None):
    N = len(trajectories)
    G_list = []
    rho_list = []
    for traj in trajectories:
        G = traj[-1]['r']  # G = r_T
        prob_e = []
        prob_b = []
        for t, transition in enumerate(traj):
            s = transition['s']
            a = transition['a']
            prob_e.append(π_e[s,a])
            prob_b.append(π_b[s,a])
        prob_e = np.array(prob_e)
        prob_b = np.array(prob_b)
        rho = (prob_e / prob_b).prod()
        G_list.append(G)
        rho_list.append(rho)
    
    G_list = np.array(G_list)
    rho_list = np.array(rho_list)
    if clip_rho is not None:
        rho_list = np.clip(rho_list, clip_rho[0], clip_rho[1])
    
    V_IS_list = G_list * rho_list
    if clip_value is not None:
        V_IS_list = np.clip(V_IS_list, clip_value[0], clip_value[1])
    
    V_IS = np.sum(V_IS_list) / N
    return V_IS, {
        'V_IS': V_IS,
        'V_WIS': np.sum(G_list * rho_list) / np.sum(rho_list),
        'ESS_IS': np.sum(rho_list) ** 2 / np.sum(rho_list ** 2),
        'G_list': G_list,
        'rho_list': rho_list,
        'V_IS_list': G_list * rho_list,
        'V_WIS_list': G_list * rho_list / np.sum(rho_list) * N,
    }
    
def perdecision_DR_recursive(trajectories, π_e, π_b, Q_hat, V_hat, gamma, clip_rho=None, clip_value=None):
    """
    Per-decision doubly robust estimator for off-policy evaluation.
    Recursive definition based on Jiang & Li (2016), https://arxiv.org/pdf/1511.03722
    """
    N = len(trajectories)
    v_list = []
    for i, traj in enumerate(trajectories):
        G = 0
        for t in reversed(range(len(traj))):
            transition = traj[t]
            s = transition['s']
            a = transition['a']
            r = transition['r']
            done = transition['done']
            rho = π_e[s, a] / π_b[s, a]
            if clip_rho is not None:
                rho = np.clip(rho, clip_rho[0], clip_rho[1])
            if done:
                G = V_hat[s] + rho * (r - Q_hat[s, a])
            else:
                G = V_hat[s] + rho * (r + gamma * G - Q_hat[s, a])
        
        v_list.append(G)

    v_list = np.array(v_list)
    if clip_value is not None:
        v_list = np.clip(v_list, clip_value[0], clip_value[1])

    v_hat = np.sum(v_list) / N
    return v_hat, {
        'v_hat': v_hat,
        'v_list': v_list,
    }

def perdecision_DR(trajectories, π_e, π_b, Q_hat, V_hat, gamma, clip_rho=None, clip_value=None):
    """
    Per-decision doubly robust estimator for off-policy evaluation.
    Non-recursive definition based on Thomas & Brunskill (2016), https://arxiv.org/pdf/1604.00923
    """
    N = len(trajectories)
    maxH = max(len(df_traj) for df_traj in trajectories)

    rho_list = np.ones((N, maxH))
    for i, traj in enumerate(trajectories):
        for t, transition in enumerate(traj):
            s = transition['s']
            a = transition['a']
            rho = π_e[s, a] / π_b[s, a]
            if clip_rho is not None:
                rho = np.clip(rho, clip_rho[0], clip_rho[1])
            rho_list[i, t] = rho
    
    # Calculate cumulative importance ratio, rho_{1:t} for each trajectory at each timestep
    rho_cumulative = np.cumprod(rho_list, axis=1)

    # # Calculate the average cumulative importance ratio at every horizon t
    # rho_normalization = np.mean(rho_cumulative, axis=0)

    v_list = []
    for i, traj in enumerate(trajectories):
        G = 0
        for t, transition in enumerate(traj):
            s = transition['s']
            a = transition['a']
            r = transition['r']
            rho_1t = rho_cumulative[i, t]
            if t == 0:
                rho_1t_1 = 1.0
            else:
                rho_1t_1 = rho_cumulative[i, t-1]
            G = G + np.power(gamma, t) * (rho_1t * r - (rho_1t * Q_hat[s,a] - rho_1t_1 * V_hat[s]))
        v_list.append(G)
    
    v_list = np.array(v_list)
    if clip_value is not None:
        v_list = np.clip(v_list, clip_value[0], clip_value[1])
    
    v_hat = np.sum(v_list) / N
    return v_hat, {
        'v_hat': v_hat,
        'v_list': v_list,
    }
    
def perdecision_WDR(trajectories, π_e, π_b, Q_hat, V_hat, gamma, clip_rho=None, clip_value=None):
    """
    Per-decision weighted doubly robust estimator for off-policy evaluation.
    Non-recursive definition based on Thomas & Brunskill (2016), https://arxiv.org/pdf/1604.00923
    """
    N = len(trajectories)
    maxH = max(len(df_traj) for df_traj in trajectories)

    rho_list = np.ones((N, maxH))
    for i, traj in enumerate(trajectories):
        for t, transition in enumerate(traj):
            s = transition['s']
            a = transition['a']
            rho = π_e[s, a] / π_b[s, a]
            if clip_rho is not None:
                rho = np.clip(rho, clip_rho[0], clip_rho[1])
            rho_list[i, t] = rho
    
    # Calculate cumulative importance ratio, rho_{1:t} for each trajectory at each timestep
    rho_cumulative = np.cumprod(rho_list, axis=1)

    # Calculate the average cumulative importance ratio at every horizon t
    rho_normalization = np.mean(rho_cumulative, axis=0)

    v_list = []
    for i, traj in enumerate(trajectories):
        G = 0
        for t, transition in enumerate(traj):
            s = transition['s']
            a = transition['a']
            r = transition['r']
            rho_1t = rho_cumulative[i, t] / rho_normalization[t]
            if t == 0:
                rho_1t_1 = 1.0
            else:
                rho_1t_1 = rho_cumulative[i, t-1] / rho_normalization[t-1]
            G = G + np.power(gamma, t) * (rho_1t * r - (rho_1t * Q_hat[s,a] - rho_1t_1 * V_hat[s]))
        v_list.append(G)
    
    v_list = np.array(v_list)
    if clip_value is not None:
        v_list = np.clip(v_list, clip_value[0], clip_value[1])
    
    v_hat = np.sum(v_list) / N
    return v_hat, {
        'v_hat': v_hat,
        'v_list': v_list,
    }

def bootstrap_OPE_estimates(v_list, n_bootstrap=1000):
    """
    Bootstrap 95% confidence interval for OPE estimates.
    """
    N = len(v_list)
    bootstrap_estimates = []
    
    for i in range(n_bootstrap):
        bootstrap_samples = resample(v_list, replace=True, random_state=i)
        bootstrap_estimates.append(np.mean(bootstrap_samples))
    
    return bootstrap_estimates

def bootstrap_WIS(OPE_stats, n_bootstrap=1000):
    """
    Bootstrap 95% confidence interval for WIS estimates.
    """
    N = len(OPE_stats['V_WIS_list'])
    bootstrap_estimates = []
    
    for i in range(n_bootstrap):
        G_list_boot, rho_list_boot = resample(OPE_stats['G_list'], OPE_stats['rho_list'], replace=True, random_state=i)
        V_WIS_boot = np.sum(G_list_boot * rho_list_boot) / np.sum(rho_list_boot)
        bootstrap_estimates.append(V_WIS_boot)
    
    return bootstrap_estimates

def V2Q(V, gymP, nS, nA, gamma):
    Q = np.zeros((nS, nA))
    for s in range(nS):
        for a in range(nA):
            Q[s, a] = sum(p * (r + (0 if done else gamma * V[s_])) for p, s_, r, done in gymP[s][a])
    return Q  # save full Q-value matrix

def policy_eval(gymP, π, nS, nA, gamma, max_iter=None):
    V = np.zeros(nS)
    num_iter = 0
    while not max_iter or num_iter < max_iter:
        V_new = V.copy()
        for s in range(nS):
            ## V[s] = sum {a} π(a|s) sum {s', r} P[s', r | s, a] * (r + gamma * V[s'])
            Q_s = np.zeros(nA)
            for a in range(nA):
                Q_s[a] = sum(p * (r + (0 if done else gamma * V[s_])) for p, s_, r, done in gymP[s][a])
            V_new[s] = np.sum(π[s] * Q_s)
        if np.isclose(np.linalg.norm(V_new - V), 1e-10):
            break
        V = V_new
        num_iter += 1
    return V

In [54]:
with open(project_dir + 'data/env_model/gymP_shifted_mixed.pkl', 'rb') as f:
    gymP_eval = pickle.load(f)

def convert_traj_format(df_traj):
    traj = []
    for _, row in df_traj.iterrows():
        traj.append({
            "s":   int(row["s:state"]),
            "a":   int(row["a:action"]),
            "r":   float(row["r:reward"]),
            "done": bool(row["done"]),
        })
    return traj

trajectories = []
for traj_id, group in tqdm(traj_shifted_test.groupby('traj'), total=traj_shifted_test['traj'].nunique()):
    usable = True
    for _, transition in group.iterrows():
        s = int(transition['s:state']); a = int(transition['a:action'])
        if s >= nS or a >= nA or np.isclose(π_b_shifted[s, a], 0.0):
            usable = False; break
    if usable:
        trajectories.append(convert_traj_format(group))

for name, π_e in policies.items():
    if name == "π_b_shifted":
        gammas = [1.0, 0.99]
    elif name.endswith("_gamma_99"):
        gammas = [0.99]
    else:
        gammas = [1.0]

    for γ in gammas:
        print(f"\n============================")
        print(f" Evaluating {name} with γ = {γ}")
        print(f"============================")

        V_e = policy_eval(gymP_eval, π_e, nS, nA, γ)
        Q_e = V2Q(V_e, gymP_eval, nS, nA, γ)

        # --- IS & WIS (note: terminal-return IS ignores γ) ---
        V_IS, IS_stats = IS(trajectories, π_e, π_b_shifted, clip_rho=(1e-3, 1e3))
        V_WIS = IS_stats['V_WIS']
        ESS = IS_stats['ESS_IS']
        V_IS_boot = bootstrap_OPE_estimates(IS_stats['V_IS_list'], n_bootstrap=500)
        V_WIS_boot = bootstrap_WIS(IS_stats, n_bootstrap=500)

        # --- DR/WDR on the EVAL env (use same γ as above) ---
        V_DR,  DR_stats  = perdecision_DR( trajectories, π_e, π_b_shifted, Q_e, V_e, γ, clip_value=(-1, 1))
        V_WDR, WDR_stats = perdecision_WDR(trajectories, π_e, π_b_shifted, Q_e, V_e, γ, clip_value=(-1, 1))
        V_DR_boot  = bootstrap_OPE_estimates(DR_stats['v_list'],  n_bootstrap=500)
        V_WDR_boot = bootstrap_OPE_estimates(WDR_stats['v_list'], n_bootstrap=500)

        # --- print ---
        print(f"IS:   {V_IS:.4f}  (95% CI: {np.percentile(V_IS_boot, 2.5):.4f}, {np.percentile(V_IS_boot, 97.5):.4f})")
        print(f"WIS:  {V_WIS:.4f} (95% CI: {np.percentile(V_WIS_boot, 2.5):.4f}, {np.percentile(V_WIS_boot, 97.5):.4f})")
        print(f"ESS:  {ESS:.2f}")
        print(f"DR:   {V_DR:.4f}  ± {np.std(V_DR_boot):.4f} (95% CI: {np.percentile(V_DR_boot, 2.5):.4f}, {np.percentile(V_DR_boot, 97.5):.4f})")
        print(f"WDR:  {V_WDR:.4f} ± {np.std(V_WDR_boot):.4f} (95% CI: {np.percentile(V_WDR_boot, 2.5):.4f}, {np.percentile(V_WDR_boot, 97.5):.4f})")


100%|██████████| 2895/2895 [00:01<00:00, 1511.79it/s]



 Evaluating π_plus with γ = 1.0
IS:   6.0347  (95% CI: 3.5506, 9.0143)
WIS:  0.9998 (95% CI: 0.9994, 1.0000)
ESS:  17.85
DR:   0.9953  ± 0.0018 (95% CI: 0.9914, 0.9985)
WDR:  0.9980 ± 0.0010 (95% CI: 0.9957, 0.9995)

 Evaluating π_minus with γ = 1.0
IS:   6.0347  (95% CI: 3.5506, 9.0143)
WIS:  0.9998 (95% CI: 0.9994, 1.0000)
ESS:  17.85
DR:   0.9960  ± 0.0015 (95% CI: 0.9926, 0.9985)
WDR:  0.9978 ± 0.0010 (95% CI: 0.9956, 0.9993)

 Evaluating π_mixed with γ = 1.0
IS:   6.0350  (95% CI: 3.5510, 9.0146)
WIS:  0.9998 (95% CI: 0.9994, 1.0000)
ESS:  17.85
DR:   0.9955  ± 0.0017 (95% CI: 0.9918, 0.9986)
WDR:  0.9980 ± 0.0010 (95% CI: 0.9957, 0.9995)

 Evaluating π_plus_gamma_99 with γ = 0.99
IS:   11.8583  (95% CI: 7.8198, 15.6753)
WIS:  0.9985 (95% CI: 0.9953, 1.0000)
ESS:  30.91
DR:   0.8647  ± 0.0069 (95% CI: 0.8511, 0.8783)
WDR:  0.9143 ± 0.0035 (95% CI: 0.9072, 0.9208)

 Evaluating π_minus_gamma_99 with γ = 0.99
IS:   6.0346  (95% CI: 3.5505, 9.0142)
WIS:  0.9998 (95% CI: 0.9994, 1.000

π_plus_gamma_99 (+) and π_mixed_gamma_99 (mixed) beats π_minus_gamma_99 (-); all three outperforms baseline

**Checking subset relationships**

In [55]:
def compare_policies(piA, piB, nameA="πA", nameB="πB", tol=1e-12):
    """
    Compare two (750, 25) policy matrices row-by-row.
    
    For each state:
      - determine relation: equal, subset, superset, incomparable
      - list which actions differ
    """

    A_sel = piA > tol
    B_sel = piB > tol

    equal = 0
    subset = 0
    superset = 0
    incomparable = 0

    diff_states = []

    for s in range(piA.shape[0]):
        A = A_sel[s]
        B = B_sel[s]

        if np.array_equal(A, B):
            equal += 1
            continue

        elif np.all(A <= B):
            subset += 1
            relation = f"{nameA} ⊂ {nameB}"

        elif np.all(B <= A):
            superset += 1
            relation = f"{nameA} ⊃ {nameB}"

        else:
            incomparable += 1
            relation = "incomparable"

        # compute differing actions
        A_actions = set(np.where(A)[0])
        B_actions = set(np.where(B)[0])
        A_only = A_actions - B_actions
        B_only = B_actions - A_actions
        intersect = A_actions & B_actions

        diff_states.append((s, A_only, B_only, intersect, relation))

    # -------- Summary --------
    print(f"\n=== Subset/Superset Summary: {nameA} vs {nameB} ===")
    print(f"Equal states:        {equal}")
    print(f"{nameA} ⊂ {nameB}:   {subset}")
    print(f"{nameA} ⊃ {nameB}:   {superset}")
    print(f"Incomparable:        {incomparable}")
    print(f"Total differing:     {subset + superset + incomparable}")

    # -------- Detailed differences --------
    print("\n=== Detailed differences by state ===\n")
    for s, A_only, B_only, intersect, relation in diff_states:
        print(f"State {s}:  ({relation})")
        print(f"  {nameA} only actions: {sorted(A_only) if A_only else '{}'}")
        print(f"  {nameB} only actions: {sorted(B_only) if B_only else '{}'}")
        print(f"  Shared actions:      {sorted(intersect) if intersect else '{}'}")
        print("")



In [56]:
compare_policies(π_plus_gamma_99, π_plus, "π_plus_gamma_99", "π_plus")


=== Subset/Superset Summary: π_plus_gamma_99 vs π_plus ===
Equal states:        52
π_plus_gamma_99 ⊂ π_plus:   615
π_plus_gamma_99 ⊃ π_plus:   0
Incomparable:        83
Total differing:     698

=== Detailed differences by state ===

State 0:  (π_plus_gamma_99 ⊂ π_plus)
  π_plus_gamma_99 only actions: {}
  π_plus only actions: [np.int64(6), np.int64(7), np.int64(8), np.int64(10), np.int64(15)]
  Shared actions:      [np.int64(0)]

State 1:  (π_plus_gamma_99 ⊂ π_plus)
  π_plus_gamma_99 only actions: {}
  π_plus only actions: [np.int64(5), np.int64(10), np.int64(15)]
  Shared actions:      [np.int64(0)]

State 2:  (π_plus_gamma_99 ⊂ π_plus)
  π_plus_gamma_99 only actions: {}
  π_plus only actions: [np.int64(6), np.int64(7), np.int64(8), np.int64(10)]
  Shared actions:      [np.int64(0)]

State 3:  (π_plus_gamma_99 ⊂ π_plus)
  π_plus_gamma_99 only actions: {}
  π_plus only actions: [np.int64(6), np.int64(9), np.int64(14), np.int64(15)]
  Shared actions:      [np.int64(8)]

State 4:  (π_p

In [57]:
compare_policies(π_minus_gamma_99, π_minus, "π_minus_gamma_99", "π_minus")


=== Subset/Superset Summary: π_minus_gamma_99 vs π_minus ===
Equal states:        749
π_minus_gamma_99 ⊂ π_minus:   0
π_minus_gamma_99 ⊃ π_minus:   1
Incomparable:        0
Total differing:     1

=== Detailed differences by state ===

State 635:  (π_minus_gamma_99 ⊃ π_minus)
  π_minus_gamma_99 only actions: [np.int64(7)]
  π_minus only actions: {}
  Shared actions:      [np.int64(0), np.int64(10), np.int64(15)]



In [58]:
compare_policies(π_mixed_gamma_99, π_mixed, "π_mixed_gamma_99", "π_mixed")


=== Subset/Superset Summary: π_mixed_gamma_99 vs π_mixed ===
Equal states:        56
π_mixed_gamma_99 ⊂ π_mixed:   621
π_mixed_gamma_99 ⊃ π_mixed:   0
Incomparable:        73
Total differing:     694

=== Detailed differences by state ===

State 0:  (π_mixed_gamma_99 ⊂ π_mixed)
  π_mixed_gamma_99 only actions: {}
  π_mixed only actions: [np.int64(6), np.int64(7), np.int64(8), np.int64(10), np.int64(15)]
  Shared actions:      [np.int64(0)]

State 1:  (π_mixed_gamma_99 ⊂ π_mixed)
  π_mixed_gamma_99 only actions: {}
  π_mixed only actions: [np.int64(5), np.int64(10), np.int64(15)]
  Shared actions:      [np.int64(0)]

State 2:  (π_mixed_gamma_99 ⊂ π_mixed)
  π_mixed_gamma_99 only actions: {}
  π_mixed only actions: [np.int64(6), np.int64(7), np.int64(8), np.int64(10)]
  Shared actions:      [np.int64(0)]

State 3:  (π_mixed_gamma_99 ⊂ π_mixed)
  π_mixed_gamma_99 only actions: {}
  π_mixed only actions: [np.int64(6), np.int64(9), np.int64(14), np.int64(15)]
  Shared actions:      [np.int

In [59]:
compare_policies(π_plus, π_mixed, "π_plus", "π_mixed")


=== Subset/Superset Summary: π_plus vs π_mixed ===
Equal states:        745
π_plus ⊂ π_mixed:   0
π_plus ⊃ π_mixed:   5
Incomparable:        0
Total differing:     5

=== Detailed differences by state ===

State 51:  (π_plus ⊃ π_mixed)
  π_plus only actions: [np.int64(10)]
  π_mixed only actions: {}
  Shared actions:      [np.int64(5), np.int64(8)]

State 294:  (π_plus ⊃ π_mixed)
  π_plus only actions: [np.int64(10)]
  π_mixed only actions: {}
  Shared actions:      [np.int64(15)]

State 330:  (π_plus ⊃ π_mixed)
  π_plus only actions: [np.int64(7)]
  π_mixed only actions: {}
  Shared actions:      [np.int64(18)]

State 478:  (π_plus ⊃ π_mixed)
  π_plus only actions: [np.int64(5)]
  π_mixed only actions: {}
  Shared actions:      [np.int64(10), np.int64(14), np.int64(15)]

State 569:  (π_plus ⊃ π_mixed)
  π_plus only actions: [np.int64(0)]
  π_mixed only actions: {}
  Shared actions:      [np.int64(1)]



1. discounting drastically changes π_plus and π_mixed, but barely affects π_minus
2. discounted policies are nearly always subsets of undiscounted policies, with 75-85 incomparable states. Which means that discounting doesn't just remove actions, but actually changes preferences in some states.